In [1]:
import pandas as pd
import os
import boto3
from io import StringIO

In [2]:
# Fazendo a conexão com o S3 e com o meu bucket
s3 = boto3.client('s3')
bucket = "aline-hp-pb"
key = "dados.csv"

In [3]:
# Lendo o arquivo CSV diretamente do S3
obj = s3.get_object(Bucket=bucket, Key=key)
data = obj['Body'].read().decode('utf-8')
df = pd.read_csv(StringIO(data), sep=';')
df

,municipio_cod,municipio_fato,data_fato,mes,ano,risp,rmbh,tentado_consumado,qtde_vitimas
0,310160,ALFENAS,02/02/2023,2,2023,18º Departamento - Poços de Caldas,3) Interior de MG,TENTADO,1
1,310240,ALVORADA DE MINAS,08/03/2023,3,2023,14º Departamento - Curvelo,3) Interior de MG,TENTADO,1
2,310350,ARAGUARI,06/04/2023,4,2023,9º Departamento - Uberlândia,3) Interior de MG,TENTADO,1
3,310350,ARAGUARI,22/06/2023,6,2023,9º Departamento - Uberlândia,3) Interior de MG,TENTADO,1
4,310350,ARAGUARI,06/07/2023,7,2023,9º Departamento - Uberlândia,3) Interior de MG,TENTADO,1
...,...,...,...,...,...,...,...,...,...
171,310040,ACAIACA,25/05/2023,5,2023,12º Departamento - Ipatinga,3) Interior de MG,CONSUMADO,1
172,314170,MESQUITA,25/05/2023,5,2023,12º Departamento - Ipatinga,3) Interior de MG,CONSUMADO,1
173,312900,GUIRICEMA,11/06/2023,6,2023,4º Departamento - Juiz de Fora,3) Interior de MG,CONSUMADO,1
174,310320,ARACAI,14/05/2023,5,2023,19º Departamento - Sete Lagoas,3) Interior de MG,CONSUMADO,1


In [ ]:
# Excluindo colunas que não vou utilizar
df = df.drop(columns=["risp", "rmbh"])
df

,municipio_cod,municipio_fato,data_fato,mes,ano,tentado_consumado,qtde_vitimas
0,310160,ALFENAS,02/02/2023,2,2023,TENTADO,1
1,310240,ALVORADA DE MINAS,08/03/2023,3,2023,TENTADO,1
2,310350,ARAGUARI,06/04/2023,4,2023,TENTADO,1
3,310350,ARAGUARI,22/06/2023,6,2023,TENTADO,1
4,310350,ARAGUARI,06/07/2023,7,2023,TENTADO,1
...,...,...,...,...,...,...,...
171,310040,ACAIACA,25/05/2023,5,2023,CONSUMADO,1
172,314170,MESQUITA,25/05/2023,5,2023,CONSUMADO,1
173,312900,GUIRICEMA,11/06/2023,6,2023,CONSUMADO,1
174,310320,ARACAI,14/05/2023,5,2023,CONSUMADO,1


In [ ]:
# Verificando os tipos das colunas
print(df.dtypes)

municipio_cod         int64
municipio_fato       object
data_fato            object
mes                   int64
ano                   int64
tentado_consumado    object
qtde_vitimas          int64
dtype: object


In [ ]:
# Convertendo os tipos de dados para numérico e data
df["municipio_cod"] = pd.to_numeric(df["municipio_cod"], errors="coerce", downcast="integer")
df["data_fato"] = pd.to_datetime(df["data_fato"], format="%d/%m/%Y", errors="coerce")
df["mes"] = pd.to_numeric(df["mes"], errors="coerce", downcast="integer")
df["ano"] = pd.to_numeric(df["ano"], errors="coerce", downcast="integer")
df["qtde_vitimas"] = pd.to_numeric(df["qtde_vitimas"], errors="coerce", downcast="integer")

# Padronizando strings para letras maiúsculas e sem espaços extras
df["municipio_fato"] = df["municipio_fato"].str.strip().str.upper()
df["tentado_consumado"] = df["tentado_consumado"].str.strip().str.upper()

In [7]:
# Removendo possíveis linhas duplicadas
df = df.drop_duplicates()

In [8]:
# Reorganizando o índice
df = df.reset_index(drop=True)

In [9]:
# Verificando se só existe "2023" na coluna "ano" e que só "CONSUMADO" e "TENTADO" na coluna "tentado_consumado"
print(df["tentado_consumado"].unique())

print(df["ano"].unique())

['TENTADO' 'CONSUMADO']
[2023]


In [10]:
# Verificando se existe algum valor nulo
print(df.isna().sum())

municipio_cod        0
municipio_fato       0
data_fato            0
mes                  0
ano                  0
tentado_consumado    0
qtde_vitimas         0
dtype: int64


In [11]:
# Salvando o DataFrame limpo de volta no S3 em formato CSV
csv_buffer = StringIO()
df.to_csv(csv_buffer, index=False, sep=';', encoding='utf-8')
s3.put_object(Bucket=bucket, Key='dados_limpos.csv', Body=csv_buffer.getvalue())

{'ResponseMetadata': {'RequestId': 'PNZX27FCKTHD5R60',
  'HostId': 'L+X4jrXFsCpFIIfbq0Ha9nvIwCvvFsR8VPel6K41dQN346ohJWBNu7y5z1p++vkJ7k2R3+Un7qU=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'L+X4jrXFsCpFIIfbq0Ha9nvIwCvvFsR8VPel6K41dQN346ohJWBNu7y5z1p++vkJ7k2R3+Un7qU=',
   'x-amz-request-id': 'PNZX27FCKTHD5R60',
   'date': 'Sun, 28 Sep 2025 18:00:41 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"e5c28a6c0cd88555d6b03250d686bfe1"',
   'x-amz-checksum-crc32': 'QObP8w==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"e5c28a6c0cd88555d6b03250d686bfe1"',
 'ChecksumCRC32': 'QObP8w==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

**QUESTIONAMENTO 1.** Será que o dia da semana influencia no feminicídio? Verifique o número de vítimas em cada dia da semana.

In [12]:
# Garantindo que a coluna "data_fato" está no formato datetime
df["data_fato"] = pd.to_datetime(df["data_fato"], errors="coerce")

In [13]:
# Criando uma nova coluna com os dias da semana
df["dia_semana"] = df["data_fato"].dt.day_name()
df

,municipio_cod,municipio_fato,data_fato,mes,ano,tentado_consumado,qtde_vitimas,dia_semana
0,310160,ALFENAS,2023-02-02,2,2023,TENTADO,1,Thursday
1,310240,ALVORADA DE MINAS,2023-03-08,3,2023,TENTADO,1,Wednesday
2,310350,ARAGUARI,2023-04-06,4,2023,TENTADO,1,Thursday
3,310350,ARAGUARI,2023-06-22,6,2023,TENTADO,1,Thursday
4,310350,ARAGUARI,2023-07-06,7,2023,TENTADO,1,Thursday
...,...,...,...,...,...,...,...,...
171,310040,ACAIACA,2023-05-25,5,2023,CONSUMADO,1,Thursday
172,314170,MESQUITA,2023-05-25,5,2023,CONSUMADO,1,Thursday
173,312900,GUIRICEMA,2023-06-11,6,2023,CONSUMADO,1,Sunday
174,310320,ARACAI,2023-05-14,5,2023,CONSUMADO,1,Sunday


In [14]:
# Traduzindo os dias da semana para português
traducao_dias = {
    "Monday": "segunda_feira",
    "Tuesday": "terça_feira",
    "Wednesday": "quarta_feira",
    "Thursday": "quinta_feira",
    "Friday": "sexta_feira",
    "Saturday": "sábado",
    "Sunday": "domingo"
}
df["dia_semana"] = df["dia_semana"].replace(traducao_dias)
df.head(10)

,municipio_cod,municipio_fato,data_fato,mes,ano,tentado_consumado,qtde_vitimas,dia_semana
0,310160,ALFENAS,2023-02-02,2,2023,TENTADO,1,quinta_feira
1,310240,ALVORADA DE MINAS,2023-03-08,3,2023,TENTADO,1,quarta_feira
2,310350,ARAGUARI,2023-04-06,4,2023,TENTADO,1,quinta_feira
3,310350,ARAGUARI,2023-06-22,6,2023,TENTADO,1,quinta_feira
4,310350,ARAGUARI,2023-07-06,7,2023,TENTADO,1,quinta_feira
5,310560,BARBACENA,2023-06-15,6,2023,TENTADO,1,quinta_feira
6,310620,BELO HORIZONTE,2023-01-01,1,2023,TENTADO,1,domingo
7,310620,BELO HORIZONTE,2023-01-06,1,2023,TENTADO,1,sexta_feira
8,310620,BELO HORIZONTE,2023-01-14,1,2023,TENTADO,1,sábado
9,310620,BELO HORIZONTE,2023-02-19,2,2023,TENTADO,1,domingo


In [15]:
# Agrupando e somando a quantidade de vítimas por dia da semana
vitimas_por_dia = df.groupby("dia_semana")["qtde_vitimas"].sum().reset_index()

vitimas_por_dia

,dia_semana,qtde_vitimas
0,domingo,44
1,quarta_feira,20
2,quinta_feira,32
3,segunda_feira,16
4,sexta_feira,19
5,sábado,38
6,terça_feira,11


In [16]:
# Reordenando os resultados para a ordem correta dos dias da semana

ordem_dias = ["segunda_feira", "terça_feira", "quarta_feira", "quinta_feira", "sexta_feira", "sábado", "domingo"]

vitimas_por_dia = vitimas_por_dia.set_index("dia_semana").reindex(ordem_dias).reset_index()

In [17]:
# Mostrando o resultado final
print(vitimas_por_dia)

      dia_semana  qtde_vitimas
0  segunda_feira            16
1    terça_feira            11
2   quarta_feira            20
3   quinta_feira            32
4    sexta_feira            19
5         sábado            38
6        domingo            44


Resposta: sim. De acordo com a análise realizada, existe influência dos dias da semana nos casos de feminicídio, vez que aos finais de semana existe uma maior quantidade de ocorrências em comparação com os demais dias.

**Questionamento 2.** Qual a soma de cada tipo dos eventos Tentado e Consumado no município de Belo Horizonte? A maioria dos eventos chega a ser consumado?

In [18]:
# Filtrando os resultados apenas para o município de Belo Horizonte
df_bh = df[df["municipio_fato"] == "BELO HORIZONTE"]

In [19]:
# Calculando o total de vítimas por tipo de evento
total = df_bh.groupby("tentado_consumado")["qtde_vitimas"].sum()
total

tentado_consumado
CONSUMADO     8
TENTADO      15
Name: qtde_vitimas, dtype: int8

In [20]:
# Verificando qual dos eventos tem maior incidência
tipo_predominante = total.idxmax()
tipo_predominante

'TENTADO'

In [ ]:
# Criando uma nova coluna com o resultado, se o evento é predominante ou não

df_bh["classificacao_evento"] = df_bh["tentado_consumado"].apply(
    lambda x: "Predominante" if x == tipo_predominante else "Não predominante"
)

C:\Users\josim\AppData\Local\Temp\ipykernel_4548\2130590991.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_bh["classificacao_evento"] = df_bh["tentado_consumado"].apply(


Resposta: no caso dos crimes tentados, temos 15 ocorrências em Belo Horizonte. Já os consumados somam 8. Dessa forma, não, nem todos os crimes chegam a ser consumados, existe predominância da tentativa.

**Questionamento 3.** Quais os cinco municípios com mais vítimas? Considere apenas os registros consumados.

In [22]:
# Filtrando apenas registros consumados
df_consumado = df[df["tentado_consumado"] == "CONSUMADO"]
df_consumado

,municipio_cod,municipio_fato,data_fato,mes,ano,tentado_consumado,qtde_vitimas,dia_semana
85,310160,ALFENAS,2023-05-06,5,2023,CONSUMADO,1,sábado
86,310350,ARAGUARI,2023-01-02,1,2023,CONSUMADO,1,segunda_feira
87,310350,ARAGUARI,2023-05-05,5,2023,CONSUMADO,1,sexta_feira
88,310420,ARCOS,2023-06-14,6,2023,CONSUMADO,1,quarta_feira
89,310560,BARBACENA,2023-07-16,7,2023,CONSUMADO,1,domingo
...,...,...,...,...,...,...,...,...
171,310040,ACAIACA,2023-05-25,5,2023,CONSUMADO,1,quinta_feira
172,314170,MESQUITA,2023-05-25,5,2023,CONSUMADO,1,quinta_feira
173,312900,GUIRICEMA,2023-06-11,6,2023,CONSUMADO,1,domingo
174,310320,ARACAI,2023-05-14,5,2023,CONSUMADO,1,domingo


In [23]:
# Agrupando por município e somando a quantidade de vítimas
totais_por_municipio = (
    df_consumado.groupby("municipio_fato")["qtde_vitimas"].sum().reset_index()
)

In [24]:
# Ordenando os municípios pelo total de vítimas em ordem decrescente
totais_ordenados = totais_por_municipio.sort_values(
    by="qtde_vitimas", ascending=False
)

In [25]:
# Selecionando apenas os cinco municípios com mais vítimas e exibindo o resultado
top5 = totais_ordenados.head(5)
print(top5)

    municipio_fato  qtde_vitimas
8   BELO HORIZONTE             8
20        CONTAGEM             5
54        PARACATU             4
9            BETIM             4
33        IPATINGA             3


Resposta: os cinco municípios com maior número de vítimas, considerando apenas eventos consumados são: Belo Horizonte, Contagem, Paracatu, Betim e Ipatinga.

In [44]:
respostas = """
QUESTIONAMENTO 1. Será que o dia da semana influencia no feminicídio? Verifique o número de vítimas em cada dia da semana.
Sim. De acordo com a análise realizada, existe influência dos dias da semana nos casos de feminicídio, vez que aos finais de semana existe uma maior quantidade de ocorrências em comparação com os demais dias.

QUESTIONAMENTO 2. Qual a soma de cada tipo dos eventos Tentado e Consumado no município de Belo Horizonte? A maioria dos eventos chega a ser consumado?
No caso dos crimes tentados, temos 15 ocorrências em Belo Horizonte. Já os consumados somam 8. Dessa forma, não, nem todos os crimes chegam a ser consumados, existe predominância da tentativa.

QUESTIONAMENTO 3. Quais os cinco municípios com mais vítimas? Considere apenas os registros consumados.
Os cinco municípios com maior número de vítimas, considerando apenas eventos consumados são: Belo Horizonte, Contagem, Paracatu, Betim e Ipatinga.
"""

s3.put_object(
    Bucket=bucket,
    Key='respostas_analises.txt', 
    Body=respostas.encode('utf-8') 
)

{'ResponseMetadata': {'RequestId': '244P7N8F41NZR0J1',
  'HostId': '5AvUde22xXSc3dvuANj+QVvIfHKfmMXoLYGAmgriKUdxtDiuU+uJGonN2tgZ31DT+26tU0dYMqI=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '5AvUde22xXSc3dvuANj+QVvIfHKfmMXoLYGAmgriKUdxtDiuU+uJGonN2tgZ31DT+26tU0dYMqI=',
   'x-amz-request-id': '244P7N8F41NZR0J1',
   'date': 'Fri, 26 Sep 2025 14:25:01 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"bb03ff218412fdbd18235aa7165cf99d"',
   'x-amz-checksum-crc32': 'y4i97w==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"bb03ff218412fdbd18235aa7165cf99d"',
 'ChecksumCRC32': 'y4i97w==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}